# SGBR B1: kiểm tra độ trung thực (fidelity gate) trên Kaggle CPU

Notebook này chạy **B1** (task C-21, đã được duyệt ngày 2026-11-21) chỉ bằng một lần **Run All**:

1. Lấy mã dự án từ GitHub private (Kaggle Secret `GITHUB_TOKEN`, quyền **Contents: Read**).
2. Cài JDK 11 + Maven đã kiểm checksum, lấy mã MOA của tác giả ở commit `f62f7b4`, build jar (hoặc dùng lại jar đã build trong session).
3. Chạy 120 run MOA nối tiếp: 6 phương pháp × {abalone, bike, elevators} × seed 1–5, lặp lại seed 1, và FIRT-DD/HTR chạy 2 lần.
4. So từng run với kết quả gốc của tác giả (`exp/post_hoc_new`), rồi in kết luận **Exact / Tolerance pass / Fail**.

**Cài đặt Kaggle:** Accelerator → **None**, bật **Internet**, gắn Secret `GITHUB_TOKEN`. Ước tính khoảng 10 phút build và khoảng 15–25 phút chạy (σ ≈ 0,84 ở B0).

**Giữ kết quả:** dùng **Save Version → Run All**, hoặc tải `/kaggle/working/B1_results.zip` ở cuối. Nếu session bị ngắt, chạy lại với cùng thư mục `results/`: runner bỏ qua các run đã xong.

Dòng B1-d (NZEnergy, đã chạy ở B0) cần file `evaluation.csv` của B0. Nếu không gắn file này, B1-d để *pending* và kết luận tổng sẽ là *Incomplete*. Khi đó, xem kết quả của 120 run trong bảng tóm tắt; B1-d đã được ghi là Exact trong repo.


In [ ]:
from pathlib import Path
import time

STARTED = time.monotonic()
GITHUB_REF = "feat/c21-b1-fidelity-gate"  # Branch containing scripts/b1_fidelity.py
GITHUB_REFRESH = True
B0_CSV = None  # Optional: path to the B0 evaluation.csv in a Kaggle Dataset input
SESSION_BUDGET_SECONDS = 10 * 3600  # Stop launching new cells after this budget
CELL_TIMEOUT_SECONDS = 1800  # Per-cell cap; the largest B1 cell took ~21 s on the authors' machine


In [ ]:
import os
import re
import subprocess
from tempfile import TemporaryDirectory

PROJECT_URL = "https://github.com/AIVIETNAM-AIO-DucTam/SGBR-Reproduction.git"
DEFAULT_CHECKOUT = Path("/kaggle/working/SGBR-Reproduction")


def project_git(folder, *args, env=None):
    try:
        return subprocess.run(
            ["git", "-C", str(folder), *args], capture_output=True, text=True,
            env=env, timeout=180, check=False,
        )
    except subprocess.TimeoutExpired:
        raise RuntimeError("Git timed out. Enable Internet in Kaggle and check GitHub connectivity.") from None
    except OSError:
        raise RuntimeError("Could not start Git. Check that Git is available in the notebook.") from None


def project_commit(folder):
    if not (folder / ".git").exists():
        return None
    head = project_git(folder, "rev-parse", "--verify", "HEAD^{commit}")
    if head.returncode or not re.fullmatch(r"[0-9a-f]{40}", head.stdout.strip()):
        raise RuntimeError("Could not resolve the project checkout's Git commit.")
    return head.stdout.strip()


def fetch_project(folder, revision):
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError(
            "Cannot read Kaggle Secret GITHUB_TOKEN. Add the fine-grained token in "
            "Kaggle Secrets and grant this notebook access to it."
        ) from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError("Kaggle Secret GITHUB_TOKEN is empty. Set it in Kaggle Secrets.")

    with TemporaryDirectory(prefix="sgbr-git-auth-") as temporary:
        askpass = Path(temporary) / "askpass"
        askpass.write_text(
            '#!/bin/sh\ncase "$1" in\n'
            '  "Username for \'https://github.com\': "*) printf "%s\\n" x-access-token ;;\n'
            '  "Password for \'https://x-access-token@github.com\': "*) '
            'printf "%s\\n" "$SGBR_GIT_SECRET" ;;\n'
            '  *) exit 1 ;;\nesac\n',
            encoding="utf-8",
        )
        askpass.chmod(0o700)
        env = {
            key: value for key, value in os.environ.items()
            if not key.startswith("GIT_") and key != "SSLKEYLOGFILE"
        }
        env.update(
            LC_ALL="C", GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0",
            GIT_CONFIG_NOSYSTEM="1", GIT_CONFIG_GLOBAL=os.devnull, SGBR_GIT_SECRET=token,
        )
        fetch = project_git(
            folder, "-c", "credential.helper=", "-c", "http.followRedirects=false",
            "-c", "http.sslVerify=true", "fetch", "--quiet", "--depth=1", "--no-tags",
            "--", PROJECT_URL, revision, env=env,
        )
    if not fetch.returncode:
        return

    # Classify the captured diagnostic without replaying Git or server output.
    diagnostic = fetch.stderr.lower()
    if any(message in diagnostic for message in (
        "could not resolve", "failed to connect", "couldn't connect", "network is unreachable",
        "connection timed out", "connect tunnel failed", "proxy", "connection reset",
    )):
        raise RuntimeError("Could not reach GitHub. Enable Internet in Kaggle notebook settings and retry.")
    if any(message in diagnostic for message in (
        "authentication failed", "repository not found", "401", "403",
        "access denied", "could not read username", "could not read password", "invalid username or token",
    )):
        raise RuntimeError(
            "GitHub authentication or repository access failed. Check that GITHUB_TOKEN is valid "
            "and has Contents: Read access to AIVIETNAM-AIO-DucTam/SGBR-Reproduction."
        )
    if any(message in diagnostic for message in ("couldn't find remote ref", "not our ref", "unadvertised object")):
        raise RuntimeError("GitHub could not fetch GITHUB_REF. Check the branch name or full commit SHA.")
    raise RuntimeError("GitHub fetch failed. Verify Internet, token permissions, and GITHUB_REF, then retry.")


def prepare_project_checkout(target, ref, refresh=False):
    if not target.parent.is_dir():
        raise RuntimeError("Automatic checkout requires /kaggle/working. Run in Kaggle or set CHECKOUT.")
    if not isinstance(ref, str) or not ref or ref.startswith("-"):
        raise ValueError("Set GITHUB_REF to a branch name or full 40-character commit SHA.")
    is_sha = re.fullmatch(r"[0-9a-fA-F]{40}", ref) is not None
    revision = ref.lower() if is_sha else f"refs/heads/{ref}"
    if not is_sha and project_git(target.parent, "check-ref-format", revision).returncode:
        raise ValueError("Set GITHUB_REF to a valid branch name or full 40-character commit SHA.")

    if target.exists() or target.is_symlink():
        if target.is_symlink() or not (target / ".git").is_dir():
            raise RuntimeError("Automatic checkout path is occupied. Set CHECKOUT to use it explicitly.")
        origin = project_git(target, "config", "--local", "--get", "remote.origin.url")
        if origin.returncode or origin.stdout.strip() != PROJECT_URL:
            raise RuntimeError("Automatic checkout path has a different Git origin. Set CHECKOUT to use it explicitly.")
        status = project_git(target, "status", "--porcelain")
        if status.returncode or status.stdout.strip():
            raise RuntimeError("Automatic checkout has local changes. Preserve them or select an explicit CHECKOUT.")
        head = project_commit(target)
        cached_ref = f"refs/remotes/origin/{ref}"
        cached = head == revision if is_sha else (
            project_git(target, "rev-parse", "--verify", cached_ref).stdout.strip() == head
        )
        if cached and not refresh:
            return head
        return checkout_project_revision(target, ref, revision, is_sha)

    # Build a shallow checkout in a sibling directory and publish it only on success.
    with TemporaryDirectory(prefix=".sgbr-checkout-", dir=target.parent) as temporary:
        folder = Path(temporary) / "repo"
        folder.mkdir()
        initial = project_git(folder, "init", "--quiet")
        if initial.returncode or project_git(folder, "remote", "add", "origin", PROJECT_URL).returncode:
            raise RuntimeError("Could not initialize the private checkout in /kaggle/working.")
        head = checkout_project_revision(folder, ref, revision, is_sha)
        folder.rename(target)
        return head


def checkout_project_revision(folder, ref, revision, is_sha):
    fetch_project(folder, revision)
    fetched = project_git(folder, "rev-parse", "--verify", "FETCH_HEAD^{commit}")
    sha = fetched.stdout.strip()
    if fetched.returncode or not re.fullmatch(r"[0-9a-f]{40}", sha) or (is_sha and sha != revision):
        raise RuntimeError("GitHub did not return the requested project commit.")
    if project_git(folder, "-c", "core.hooksPath=/dev/null", "checkout", "--quiet", "--detach", sha).returncode:
        raise RuntimeError("Could not check out the fetched project commit.")
    if not is_sha and project_git(folder, "update-ref", f"refs/remotes/origin/{ref}", sha).returncode:
        raise RuntimeError("Could not record the fetched project branch.")
    return project_commit(folder)


In [ ]:
import datetime
import sys

WORKING = Path("/kaggle/working")
if not WORKING.is_dir():
    raise RuntimeError("Open this notebook in a Kaggle CPU session with /kaggle/working available.")
print(f"[{datetime.datetime.now(datetime.timezone.utc):%H:%M:%S}] Resolving the private project checkout", flush=True)
refresh = GITHUB_REFRESH and re.fullmatch(r"[0-9a-fA-F]{40}", GITHUB_REF) is None
commit = prepare_project_checkout(DEFAULT_CHECKOUT, GITHUB_REF, refresh)
SCRIPTS = DEFAULT_CHECKOUT / "scripts"
for name in ("b1_fidelity", "sgbr_c01_e1_integrated", "e1_memory_feasibility", "kaggle_environment_smoke"):
    if not (SCRIPTS / f"{name}.py").is_file():
        raise FileNotFoundError(f"{SCRIPTS / f'{name}.py'} is missing. Select a GITHUB_REF that contains the B1 runner.")
print(f"Project commit: {commit}", flush=True)

sys.dont_write_bytecode = True
sys.path.insert(0, str(SCRIPTS))
for name in ("sgbr_c01_e1_integrated", "e1_memory_feasibility", "kaggle_environment_smoke"):
    sys.modules.pop(name, None)
import sgbr_c01_e1_integrated as pipeline

RESULTS = WORKING / "results"
ARTIFACTS = RESULTS / "artifacts"
CACHE = WORKING / ".sgbr-cache"
for directory in (ARTIFACTS, CACHE):
    directory.mkdir(parents=True, exist_ok=True)

pipeline.stage("Provisioning JDK 11 and Maven")
JAVA, MVN, TOOLS = pipeline.provision(CACHE)
pipeline.stage("Fetching the authors' pinned MOA source")
UPSTREAM, PIN = pipeline.upstream_checkout(CACHE)
pipeline.stage("Building or reusing the MOA jar (first build takes several minutes)")
JAR, BUILD = pipeline.build_or_reuse(UPSTREAM, ARTIFACTS, MVN, TOOLS, PIN)
PROVENANCE = ARTIFACTS / "moa_build.json"
print(f"Jar: {JAR}\nSHA-256: {BUILD['sha256']} (reused: {BUILD['reused']})", flush=True)


In [ ]:
import subprocess

session_id = "kaggle-b1-" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
base = [sys.executable, str(SCRIPTS / "b1_fidelity.py"), "--upstream-checkout", str(UPSTREAM), "--results", str(RESULTS)]
if B0_CSV:
    base += ["--b0-csv", str(B0_CSV)]


def stream(arguments):
    process = subprocess.Popen(
        arguments, cwd=DEFAULT_CHECKOUT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    return process.wait()


if stream(base + ["manifest"]):
    raise RuntimeError("Could not generate the B1 cell manifest.")
pipeline.stage(f"Launching B1 (session {session_id})")
exit_code = stream(base + [
    "run", "--approved", "--java", str(JAVA), "--moa-jar", str(JAR), "--jar-provenance", str(PROVENANCE),
    "--session-id", session_id, "--session-budget-seconds", str(SESSION_BUDGET_SECONDS),
    "--cell-timeout-seconds", str(CELL_TIMEOUT_SECONDS),
])
print(f"B1 runner exit code: {exit_code}", flush=True)


In [ ]:
import csv
import json
import shutil
from collections import Counter

gate = json.loads((RESULTS / "summary/B1_gate.json").read_text(encoding="utf-8"))
rows = list(csv.DictReader((RESULTS / "summary/B1_parity.csv").open(encoding="utf-8")))
print("Overall verdict:", gate.get("verdict"))
print("Completed runs:", gate.get("completed_b1_runs"), "/", gate.get("scheduled_b1_runs"))
if gate.get("blockers"):
    print("Blockers:", gate["blockers"])
print("Verdict by part:")
for (part, verdict), count in sorted(Counter((row["exp"], row["verdict"]) for row in rows).items()):
    print(f"  {part:8s} {verdict:16s} {count}")
deltas = [abs(float(row["delta"])) for row in rows if row["exp"] == "B1-a" and row["delta"]]
if deltas:
    print(f"B1-a max |delta R2| vs authors: {max(deltas):.3e} over {len(deltas)} runs")
for group in gate.get("serial_slowdown_by_cpu", []):
    print("Slowdown sigma:", group)

archive = shutil.make_archive(str(WORKING / "B1_results"), "zip", RESULTS)
print(f"Download {archive} (contains raw/, summary/, logs/, manifest/).")
